In [2]:
# pip install paramiko
import random
import os
import telnetlib
import time
import datetime
import json
import paramiko
import ftplib
from contextlib import closing
import time

def Input_cmd_func(input_cmd_file_name):
    # Get command from txt
    # input_cmd_file_name = 'input_cmd.txt'    
    print("",file = open(input_cmd_file_name, '+a'))    
    print("ENDDD",file = open(input_cmd_file_name, '+a'))    
    f = open(input_cmd_file_name,'r')
    line = f.readline().replace('\n','')
    lines = []
    My_List = []
    while (line != 'ENDDD'):
        if(line!=''):           
            My_List.append(line)
        lines.append(line+"\n")
        line = f.readline().replace('\n','')    
    return My_List

def Telnet_func(host_ip,My_List,output_file):    
    username = 'admin'
    password = '1234'
    print(f'### telnet {host_ip}',file = open(output_file, '+a'))
    tnet_hndl = telnetlib.Telnet(host_ip)
    print (tnet_hndl.read_until(b"login: "),file = open(output_file, '+a'))
    tnet_hndl.write(username.encode('ascii') + b"\n")
    print (tnet_hndl.read_until(b"Password: "),file = open(output_file, '+a'))
    tnet_hndl.write(password.encode('ascii') + b"\n")
    print (tnet_hndl.read_until(b"# "),file = open(output_file, '+a'))
    tnet_hndl.write(b"config" + b"\n")
    print (tnet_hndl.read_until(b"# "),file = open(output_file, '+a'))
    for cmd in My_List:
        cmd = bytes(cmd, encoding='UTF-8')
        print(cmd,file = open(output_file, '+a'))  # print command
        tnet_hndl.write(cmd + b"\n")
        list_1 = str(tnet_hndl.read_until(b"# ")).split('\\r\\n')
        for i in list_1:
            print (f':',i,file = open(output_file, '+a'))  # print return data
    tnet_hndl.close()

def channel_exe(channel, cmd, t=0.2):
    channel.send(cmd)
    channel.send('\n')
    time.sleep(t)
    resp = channel.recv(9999).decode("utf8")
    print(cmd)
    print(resp)
    return resp

# START Main Func ====================================================================================================


# output_file = f'output_{str(datetime.datetime.now()).split(":")[0]}{str(datetime.datetime.now()).split(":")[1]}.txt'
output_file = f'output_command.txt'
file = open(output_file,'w+')


# fetch data ======================
with open('data.json') as f:
    data = json.load(f)
telnet_ip = data['telnet_ip']
username = data['username']
password = data['password']
ssh_ip = data['ssh_ip']
ssh_password = data['ssh_password']
# =================================

# Telnet -----------------------------------------------------------------------------------------
print('Telnet')
host_ip = telnet_ip  
input_cmd_file_name = 'input_cmd.txt'
My_List = Input_cmd_func(input_cmd_file_name)
Telnet_func(host_ip,My_List,output_file)

# Ssh ---------------------------------------------------------------------------------------------
print('SSH')
print('\nSSH\n',file = open(output_file, '+a')) 
# cmd = ['cd /opt/ELEMENTVIEW_MASTER/bin/',
#        'ls'           
#        ]
ssh = paramiko.SSHClient()
ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
ssh.connect(hostname=ssh_ip, username=username, password=ssh_password)

# com = 'sudo su'
channel_ssh = ssh.invoke_shell()

com = f'cd /opt/ELEMENTVIEW_MASTER/bin/'
channel_exe(channel_ssh, com)
com = f'ls'
channel_exe(channel_ssh, com)
com = 'sudo su'
if(f"password for {username}" in channel_exe(channel_ssh, com)):
    print('111')
    com = ssh_password
    channel_exe(channel_ssh, com)
    print('111')
com = f'chmod 777 {telnet_ip}'
channel_exe(channel_ssh, com)
com = f'ls'
channel_exe(channel_ssh, com)

# channel_ssh.close()
# ssh.close()



# Ftp -----------------------------------------------------------------------------
print('FTP')
print (f'\nFTP\n',file = open(output_file, '+a'))
count = 0
if_pass = False
while((if_pass == False) and count < 1):
    with closing(ftplib.FTP(ssh_ip)) as ftp:

        
        try:
            # ftp.set_debuglevel(2)
            ftp.login(username,ssh_password)
            
            ftp.cwd('/opt/ELEMENTVIEW_MASTER/bin')
            ftp.retrlines('LIST')
            
            localfile = open('d:/'+telnet_ip+'.json','wb')    # 本地端要儲存的檔案
            ftp.retrbinary('RETR ' + telnet_ip, localfile.write)    # 下載檔案

        except ftplib.error_perm as error:
            print (f'*** ftp Fail ***   ',error,file = open(output_file, '+a'))
            print (error)

    try:
        ftp.quit()
    except:
        pass
    try:
        ftp.close()
    except:
        pass

    count += 1

channel_ssh.close()
ssh.close()    



Telnet
1
SSH
cd /opt/ELEMENTVIEW_MASTER/bin/
Welcome to Ubuntu 22.04.2 LTS (GNU/Linux 5.15.0-73-generic x86_64)

 * Documentation:  https://help.ubuntu.com
 * Management:     https://landscape.canonical.com
 * Support:        https://ubuntu.com/advantage

Expanded Security Maintenance for Applications is not enabled.

111 updates can be applied immediately.
To see these additional updates run: apt list --upgradable

9 additional security updates can be applied with ESM Apps.
Learn more about enabling ESM Apps service at https://ubuntu.com/esm

*** System restart required ***
Last login: Mon Oct 23 11:35:24 2023 from 192.168.172.91
admin@PQAEMS:~$ cd /opt/ELEMENTVIEW_MASTER/bin/
admin@PQAEMS:/opt/ELEMENTVIEW_MASTER/bin$ 
ls
ls
192.168.169.86                        l2backup.jar
665e6123-cd03-4630-922f-46763d829dac  masterems
apache-tomcat                         nm880abackup.jar
apache-tomcat-axis2                   nm880arestore.jar
apache-tomcat-RESTful                 redundantServer


In [6]:
import json
import datetime
import time
import random
import os
import telnetlib
import paramiko
import ftplib
from contextlib import closing
 
# JSON_file = 'd:/192.168.169.86_01.json'
JSON_file = 'd:/192.168.169.86_1st39_2nd44.json'

def is_json(myjson):
    try:
        json.loads(myjson)
    except ValueError:
        return False
    return True

def is_json_f(file):
    try:
        with open(file) as f:
            # data = json.load(f)
            # json.loads(f)
            json.load(f)
    except ValueError:
        return False
    return True

# data = {
#     'name' : 'ACME',
#     'shares' : 100,
#     'price' : 542.23
# }
 
# json_str = json.dumps(data)

def prin():
    print (is_json_f(JSON_file))
    print (is_json_f('D:\zh_memo\Telnet_Check_project\data.json'))

def test():
    with open(JSON_file) as f:
        data = json.load(f,strict=False)
        print(data)
def dump():
    with open(JSON_file) as f:
        json.dump(f)

# prin()
# test()  # open wrd



# with open('n.json') as f:
#     json.load(f)

# f = open('input.txt','r')
# line = f.readline().replace('\n','')
# while(line):
#     print(line)
#     line = f.readline().replace('\n','')

# -----------------------------------------------------------------------------------

# fetch data ==========================
with open('data.json') as f:
    data0 = json.load(f)
telnet_ip = data0['telnet_ip']
username = data0['username']
password = data0['password']
ssh_ip = data0['ssh_ip']
ssh_password = data0['ssh_password']

with open('data_cardtype_list.json') as f:
    dict_cardtype = json.load(f)
dict_rate = {"1g-full":"1000"}
# output command ======================
# set_up_file = open("output_command.txt",'r')
# lines_env = set_up_file.readlines()
# set_up_file.close()
time1 = f'{str(datetime.datetime.now()).split(":")[0]}{str(datetime.datetime.now()).split(":")[1]}'
output_file = f'output_command_{time1}.txt'
file0 = open(output_file,'w+')
# load json ===========================
print('00')
with open(JSON_file) as f:
    data = json.load(f,strict=False)
#     print(data)

print( data['ipconf']['iplist'])
print( data['ipconf']['iplist'][0]['interface'])





# Telent and Check ======================
def f(com,tnet_hndl,output_file):
    print(com,file = open(output_file, '+a'))  # print command
    tnet_hndl.write(com.encode('utf-8') + b"\n")
    s = str(tnet_hndl.read_until(b"# "))
    list_1 = s.split('\\r\\n')
    for i in list_1:
        print (f':',i,file = open(output_file, '+a'))  # print return data
    return s

host_ip = telnet_ip
tnet_hndl = telnetlib.Telnet(host_ip)
print (tnet_hndl.read_until(b"login: "),file = open(output_file, '+a'))
tnet_hndl.write(username.encode('ascii') + b"\n")
print (tnet_hndl.read_until(b"Password: "),file = open(output_file, '+a'))
tnet_hndl.write(password.encode('ascii') + b"\n")
print (tnet_hndl.read_until(b"# "),file = open(output_file, '+a'))



def onelinetry(com,content,print_t,tnet_hndl,output_file,t_file):
    print(com,file = open(output_file, '+a'))  # print command
    tnet_hndl.write(com.encode('ascii') + b"\n")
    s = str(tnet_hndl.read_until(b"# "))
    list_1 = s.split('\\r\\n')
    for i in list_1:
        print (f':',i,file = open(output_file, '+a'))  # print return data
    # return list_1
    # return s
    if(content not in s):  # (value)
        print (f'bootfwversion Test Fail ({content})',file = open(t_file, '+a'))
    else:
        print (f'bootfwversion Test Pass ({content})',file = open(t_file, '+a'))

def printReturnMsg(msg,keyword,file_name):
    # get list
    my_list = msg.split('\\r\\n')
    if_find = False
    for i in my_list:
        if keyword in i:
            if_find = True
            print (i,file = open(file_name, '+a'))            
            break
    if(if_find == False):
        print (f'Not Found "{keyword}"',file = open(file_name, '+a'))       

def Find_title(out,title,type=0):
    my_list = out.split('\\r\\n')
    for i in my_list: ### only return the first match one
        if(f'{title}' in i):
            temp = i.split(f'{title}')[1]
            
            match type:
                case 1:                              # titile | val\r
                    temp = temp.split()[1].split('\\')[0]
                case 2:                              # titile :val
                    temp = temp.split(':')[1]
                case 3:                              # FW version {} v v
                    temp = i.split(f'{title}')[0].split()[2]                    


            # if(type == 1):                              # titile | val\r
            #     # temp = temp.split()[2].split('\\')[0]
            #     temp = temp.split()[1].split('\\')[0]
            # elif(type == 2):                            # titile :val
            #     temp = temp.split(':')[1]

            return temp
            break
    return None

# Test Start ===========================================================================================
t_file = f'test_{time1}.txt'
file = open(t_file,'w+')
blank = "             "

try:
    # out = f('config',tnet_hndl,output_file)
    # out = f('in genni 1',tnet_hndl,output_file)
    # out = f('name "xxxxxxxxxxxxxxxname_31 x"',tnet_hndl,output_file)
    # out = f('exit',tnet_hndl,output_file)
    # out = f('exit',tnet_hndl,output_file)

    if_error = False

    # Check version
    out = f('show version',tnet_hndl,output_file)    
    temp = Find_title(out,'running image : ')
    if(temp == None):
        print (f'{blank}"running image : " Not Found in return msg, bootfwversion Test Fail ',file = open(t_file, '+a'))
        if_error = True
    elif(data['bootfwversion'] != temp):       
        print (f"{blank}bootfwversion Test Fail ({data['bootfwversion']}!={temp})",file = open(t_file, '+a'))
        if_error = True
    else:
        print (f"{blank}bootfwversion Test Pass ({data['bootfwversion']})",file = open(t_file, '+a'))

    # Check image 1 version
    out = f('show lc status < MSC >',tnet_hndl,output_file)
    out = f('show lcman status 17',tnet_hndl,output_file)    
    temp = Find_title(out,'image 1 version:')
    if(temp == None):
        print (f'{blank}image 1 version Not Found in return msg, image 1 version Test Fail ',file = open(t_file, '+a'))
        if_error = True
    elif(data['fwimage1version'] != temp):                              ###
        print (f"{blank}image 1 version Test Fail ({data['fwimage1version']}!={temp})",file = open(t_file, '+a'))
        if_error = True
    else:
        print (f"{blank}image 1 version Test Pass ({data['fwimage1version']})",file = open(t_file, '+a'))        
    # Check image 2 version
    temp = Find_title(out,'image 2 version:')
    if(temp == None):
        print (f'{blank}image 2 version Not Found in return msg Test Fail ',file = open(t_file, '+a'))
        if_error = True
    elif(data['fwimage2version'] != temp):                              ###
        print (f"{blank}image 2 version Test Fail ({data['fwimage2version']}!={temp})",file = open(t_file, '+a'))
        if_error = True
    else:
        print (f"{blank}image 2 version Test Pass ({data['fwimage2version']})",file = open(t_file, '+a'))

    # Check location
    out = f('show system',tnet_hndl,output_file)
    temp = Find_title(out,'Location: ')
    if(temp == None):
        print (f'{blank}"Location: " Not Found in return msg Test Fail ',file = open(t_file, '+a'))
        if_error = True
    elif(data['location'] not in temp):
        print (f'{blank}location Test Fail ({data["location"]}!={temp})',file = open(t_file, '+a'))
        if_error = True
    else:
        print (f"{blank}location Test Pass ({data['location']})",file = open(t_file, '+a'))
    # Check contact
    temp = Find_title(out,'Contact: ')
    if(temp == None):
        print (f'{blank}"Contact: " Not Found in return msg Test Fail ',file = open(t_file, '+a'))
        if_error = True
    elif(data['contact'] not in temp):
        print (f'{blank}contact Test Fail ([{data["contact"]}!={temp})',file = open(t_file, '+a'))
        if_error = True
    else:
        print (f"{blank}contact Test Pass ({data['contact']})",file = open(t_file, '+a'))
    # Check mac address
    temp = Find_title(out,'MAC address: ')
    if(temp == None):
        print (f'{blank}"MAC address: " Not Found in return msg Test Fail ',file = open(t_file, '+a'))
        if_error = True
    elif(data['macaddress'] not in temp):
        print (f'{blank}MAC address Test Fail ({data["macaddress"]}!={temp})',file = open(t_file, '+a'))
        if_error = True
    else:
        print (f"{blank}MAC address Test Pass ({data['macaddress']})",file = open(t_file, '+a'))
 
    # Check trap community
    out = f('show snmp',tnet_hndl,output_file)              ###    
    temp = Find_title(out,'trap community: ')
    if(temp == None):
        temp = 'Not Found "trap community"'
    if(data['trapcommunity'] not in temp):
        print (f"{blank}trap community Test Fail ({data['trapcommunity']}!={temp})",file = open(t_file, '+a'))
        if_error = True
    else:
        print (f"{blank}trap community Test Pass ({temp})",file = open(t_file, '+a'))

    if(if_error == False):
        print (f"[MSAN1-16425][Device_info - System info] Result Pass",file = open(t_file, '+a'))
    else:
        print (f"[MSAN1-16425][Device_info - System info] Result Fail",file = open(t_file, '+a'))

    
    if_error = False
    # Check interface
    out = f('show ip',tnet_hndl,output_file)              ###
    my_list = out.split('\\r\\n')[4].split()
    if(data['ipconf']['iplist'][0]['interface'] != my_list[0]):
        print (f"{blank}interface Test Fail ({data['ipconf']['iplist'][0]['interface']}!={my_list[0]})",file = open(t_file, '+a'))
        if_error = True
    else:
        print (f"{blank}interface Test Pass ({my_list[0]})",file = open(t_file, '+a'))
    # Check ip
    if(data['ipconf']['iplist'][0]['ip'] != my_list[1]):
        print (f"{blank}ip Test Fail ({data['ipconf']['iplist'][0]['ip']}!={my_list[1]})",file = open(t_file, '+a'))
        if_error = True
    else:
        print (f"{blank}ip Test Pass ({my_list[1]})",file = open(t_file, '+a'))
    # Check mask
    if(data['ipconf']['iplist'][0]['mask'] != my_list[2]):
        print (f"{blank}mask Test Fail ({data['ipconf']['iplist'][0]['mask']}!={my_list[2]})",file = open(t_file, '+a'))
        if_error = True
    else:
        print (f"{blank}mask Test Pass ({my_list[2]})",file = open(t_file, '+a'))
    # Check gateway
    if(data['ipconf']['iplist'][0]['gateway'] != my_list[3]):
        print (f"{blank}gateway Test Fail ({data['ipconf']['iplist'][0]['gateway']}!={my_list[3]})",file = open(t_file, '+a'))
        if_error = True
    else:
        print (f"{blank}gateway Test Pass ({my_list[3]})",file = open(t_file, '+a'))
    # Check vid
    if(str(data['ipconf']['iplist'][0]['vid']) != str(my_list[4])):
        print (f"{blank}vid Test Fail ({data['ipconf']['iplist'][0]['vid']}!={my_list[4]})",file = open(t_file, '+a'))
        if_error = True
    else:
        print (f"{blank}vid Test Pass ({my_list[4]})",file = open(t_file, '+a'))
    if(if_error == False):
        print (f"[MSAN1-16426][Device_info - Interface - ipconf] Result Pass",file = open(t_file, '+a'))
    else:
        print (f"[MSAN1-16426][Device_info - Interface - ipconf] Result Fail",file = open(t_file, '+a'))
    
    if_error = False
    # Check slot 9(MSC), 16(GLC), 10(OLC GPON), 17(XPON)     (slot info)
    print (f'\n{blank}slotinfo ',file = open(t_file, '+a'))
    k_slot = [9, 16, 10, 17]
    for i in range(4):        
        slot = str(k_slot[i])
        idex = k_slot[i]-1
        print (f"{blank}slot :{slot}",file = open(t_file, '+a'))
        out = f(f'show lc st {slot}',tnet_hndl,output_file)              ###
        temp = out.split('\\r\\n')[1].split('slot')[1].split(':')[0]
        if(str(data['slotinfo'][idex]['slot']) not in str(temp)):
            print (f"{blank}slot {slot} Test Fail ({data['slotinfo'][idex]['slot']}!={temp})",file = open(t_file, '+a'))
            if_error = True
        else:
            print (f"{blank}slot {slot} Test Pass ({data['slotinfo'][idex]['slot']})",file = open(t_file, '+a'))
        # Check name
        temp = Find_title(out,'name  : ')
        if(temp == None):
            print (f'{blank}"name  : " Not Found in return msg Test Fail ',file = open(t_file, '+a'))
            if_error = True
        elif(str(data['slotinfo'][idex]['type']) not in str(dict_cardtype[temp.lower()])):
            print (f"{blank}name Test Fail ({data['slotinfo'][idex]['type']}!={dict_cardtype[temp.lower()]} {temp})",file = open(t_file, '+a'))
            if_error = True
        else:
            print (f"{blank}name Test Pass ({data['slotinfo'][idex]['type']} {temp})",file = open(t_file, '+a'))
        # Check hardware version
        temp = Find_title(out,'hardware version        : ')
        if(temp == None):
            print (f'{blank}"hardware version        : " Not Found in return msg Test Fail ',file = open(t_file, '+a'))
            if_error = True
        elif(str(data['slotinfo'][idex]['hwversion']) not in str(temp)):
            print (f"{blank}hardware version Test Fail ({data['slotinfo'][idex]['hwversion']}!={temp})",file = open(t_file, '+a'))
            if_error = True
        else:
            print (f"{blank}hardware version Test Pass ({data['slotinfo'][idex]['hwversion']})",file = open(t_file, '+a'))
        # Check firmware version
        temp = Find_title(out,'firmware version        : ')
        if(temp == None):
            print (f'{blank}"firmware version        : " Not Found in return msg Test Fail ',file = open(t_file, '+a'))
            if_error = True
        elif(str(data['slotinfo'][idex]['fwversion']) not in str(temp)):
            print (f"{blank}firmware version Test Fail ({data['slotinfo'][idex]['fwversion']}!={temp})",file = open(t_file, '+a'))
            if_error = True
        else:
            print (f"{blank}firmware version Test Pass ({data['slotinfo'][idex]['fwversion']})",file = open(t_file, '+a'))
        # Check hardware serial number
        temp = Find_title(out,'hardware serial number  : ')
        if(temp == None):
            print (f'{blank}"hardware serial number  : " Not Found in return msg Test Fail ',file = open(t_file, '+a'))
            if_error = True
        elif(str(data['slotinfo'][idex]['serialnumber']) not in str(temp)):
            print (f"{blank}hardware serial number Test Fail ({data['slotinfo'][idex]['serialnumber']}!={temp})",file = open(t_file, '+a'))
            if_error = True
        else:
            print (f"{blank}hardware serial number Test Pass ({data['slotinfo'][idex]['serialnumber']})",file = open(t_file, '+a'))
        print (f"{blank}...operstatus...",file = open(t_file, '+a'))
        print (f"{blank}...adminstate...",file = open(t_file, '+a'))
        print (f"{blank}...alarmstate...",file = open(t_file, '+a'))
    if(if_error == False):
        print (f"[MSAN1-16427][Device_info - Interface - slotinfo] Result Pass",file = open(t_file, '+a'))
    else:
        print (f"[MSAN1-16427][Device_info - Interface - slotinfo] Result Fail",file = open(t_file, '+a'))

    # GENNI (port info)
    print (f'\n{blank}portinfo ',file = open(t_file, '+a'))
    if_error = False
    k_slot = [9, 16, 10, 17]   # 9-1, 16-1, 10-16, 17-16
    k_idex = [0, 64, 21,120]
    for i in range(4): 
        out = f('show lc status < MSC >',tnet_hndl,output_file)     
        slot = str(k_slot[i])
        idex = k_idex[i]
        out = f(f'show lcman status {slot}',tnet_hndl,output_file)  ####  MSC slot
        # Check slot
        temp = out.split('\\r\\n')[1].split('slot')[1].split(':')[0]
        if(str(data['portinfo'][idex]['slot']) not in str(temp)):
            print (f"{blank}slot {slot} Test Fail ({data['portinfo'][idex]['slot']}!={temp})",file = open(t_file, '+a'))
            if_error = True
        else:
            print (f"{blank}slot {slot} Test Pass ({data['portinfo'][idex]['slot']})",file = open(t_file, '+a'))
        # Check name
        temp = Find_title(out,'name  : ')
        if(temp == None):
            print (f'{blank}"name  : " Not Found in return msg Test Fail ',file = open(t_file, '+a'))
            if_error = True
        elif(str(data['portinfo'][idex]['name']) != str(dict_cardtype[temp.lower()])):
            print (f"{blank}name Test Fail ({data['portinfo'][idex]['name']}!={dict_cardtype[temp.lower()]} {temp})",file = open(t_file, '+a'))
            if_error = True
        else:
            print (f"{blank}name Test Pass ({str(data['portinfo'][idex]['name'])})",file = open(t_file, '+a'))        
        # Check port
        out = f('show interface genni 1 status',tnet_hndl,output_file)          ###
        temp = out.split('\\r\\n')[4].split()[0]
        if(str(data['portinfo'][idex]['port']) not in str(temp)):
            print (f"{blank}port Test Fail ({data['portinfo'][idex]['port']}!={temp})",file = open(t_file, '+a'))
            if_error = True
        else:
            print (f"{blank}port Test Pass ({data['portinfo'][idex]['port']})",file = open(t_file, '+a'))
        print (f"{blank}...type...",file = open(t_file, '+a'))
        temp = out.split('\\r\\n')[4].split()[2]
        if(str(data['portinfo'][idex]['operstatus']) not in str(temp)):
            print (f"{blank}operstatus Test Fail ({data['portinfo'][idex]['operstatus']}!={temp})",file = open(t_file, '+a'))
            if_error = True
        else:
            print (f"{blank}operstatus Test Pass ({data['portinfo'][idex]['operstatus']})",file = open(t_file, '+a'))
        temp = out.split('\\r\\n')[4].split()[3]
        if(str(data['portinfo'][idex]['rate']) not in str(dict_rate[temp])):
            print (f"{blank}rate Test Fail ({data['portinfo'][idex]['rate']}!={dict_rate[temp]})",file = open(t_file, '+a'))
            if_error = True
        else:
            print (f"{blank}rate Test Pass ({data['portinfo'][idex]['rate']})",file = open(t_file, '+a'))        
        
        out = f('show ru interface genni 1',tnet_hndl,output_file)              ###
        # Check adminstate ...
        # Check linetemplate ...
        # Check alarmtemplate ...
        # Check upstreamrate ...
        # Check downstreamrate ...
        # Check fiberstatus ...
        print (f"{blank}...adminstate...",file = open(t_file, '+a'))
        print (f"{blank}...linetemplate...",file = open(t_file, '+a'))
        print (f"{blank}...alarmtemplate...",file = open(t_file, '+a'))
        print (f"{blank}...upstreamrate...",file = open(t_file, '+a'))
        print (f"{blank}...downstreamrate...",file = open(t_file, '+a'))
        print (f"{blank}...fiberstatus...",file = open(t_file, '+a'))
    if(if_error == False):
        print (f"[MSAN1-16428][Device_info - Interface - portinfo] Result Pass",file = open(t_file, '+a'))
    else:
        print (f"[MSAN1-16428][Device_info - Interface - portinfo] Result Fail",file = open(t_file, '+a'))


    if_error = False
    if(if_error == False):
        print (f"[MSAN1-16429][Device_info - Interface - ontinfo] Result Pass",file = open(t_file, '+a'))
    else:
        print (f"[MSAN1-16429][Device_info - Interface - ontinfo] Result Fail",file = open(t_file, '+a'))



    
    # (xontinfo)
    print (f'\n{blank}xontinfo ',file = open(t_file, '+a'))
    if_error = False
    slot = '17'
    port='1'
    ont = '128'
    out = f('sh in remote x 17-1-128 status',tnet_hndl,output_file) 
    # Check slot
    temp = Find_title(out,'AID',1)             ###
    if(str(data['xontinfo'][0]['slot']) != temp.split('-')[0]):
        print (f"{blank}slot Test Fail ({data['xontinfo'][0]['slot']}!={temp.split('-')[0]})",file = open(t_file, '+a'))
        if_error = True
    else:
        print (f"{blank}slot Test Pass ({data['xontinfo'][0]['slot']})",file = open(t_file, '+a'))
    # Check port
    if(str(data['xontinfo'][0]['port']) != temp.split('-')[1]):
        print (f"{blank}port Test Fail ({data['xontinfo'][0]['port']}!={temp.split('-')[1]})",file = open(t_file, '+a'))
        if_error = True
    else:
        print (f"{blank}port Test Pass ({data['xontinfo'][0]['port']})",file = open(t_file, '+a'))
    # Check ont
    if(str(data['xontinfo'][0]['ontid']) != temp.split('-')[2]):
        print (f"{blank}ontid Test Fail ({data['xontinfo'][0]['ontid']}!={temp.split('-')[2]})",file = open(t_file, '+a'))
        if_error = True
    else:
        print (f"{blank}ontid Test Pass ({data['xontinfo'][0]['ontid']})",file = open(t_file, '+a'))
    # Check sn
    temp = Find_title(out,'SN',1)
    if(temp == None):
        print (f'{blank}"SN" Not Found in return msg Test Fail ',file = open(t_file, '+a'))
        if_error = True
    elif(str(data['xontinfo'][0]['sn']) != str(temp)):
        print (f"{blank}sn Test Fail ({data['xontinfo'][0]['sn']}!={temp})",file = open(t_file, '+a'))
        if_error = True
    else:
        print (f"{blank}sn Test Pass ({str(data['xontinfo'][0]['sn'])})",file = open(t_file, '+a'))
    # Check Registration-Id 
    temp = Find_title(out,'Registration-Id',1)
    if(temp == None):
        print (f'{blank}"Registration-Id" Not Found in return msg Test Fail ',file = open(t_file, '+a'))
        if_error = True
    elif(str(data['xontinfo'][0]['password']) != str(temp)):
        print (f"{blank}Registration-Id Test Fail ({data['xontinfo'][0]['password']}!={temp})",file = open(t_file, '+a'))
        if_error = True
    else:
        print (f"{blank}Registration-Id Test Pass ({str(data['xontinfo'][0]['password'])})",file = open(t_file, '+a'))
    # Check Status
    temp = Find_title(out,'Status',1)
    if(temp == None):
        print (f'{blank}"Status" Not Found in return msg Test Fail ',file = open(t_file, '+a'))
        if_error = True
    elif(str(data['xontinfo'][0]['ontstate']) != str(temp)):
        print (f"{blank}ontstate Test Fail ({data['xontinfo'][0]['ontstate']}!={temp})",file = open(t_file, '+a'))
        if_error = True
    else:
        print (f"{blank}ontstate Test Pass ({str(data['xontinfo'][0]['ontstate'])})",file = open(t_file, '+a'))
    # Check vendorid
    temp = Find_title(out,'Vendor ID',1)
    if(temp == None):
        print (f'{blank}"Vendor ID" Not Found in return msg Test Fail ',file = open(t_file, '+a'))
        if_error = True
    elif(str(data['xontinfo'][0]['vendorid']) != str(temp)):
        print (f"{blank}vendorid Test Fail ({data['xontinfo'][0]['vendorid']}!={temp})",file = open(t_file, '+a'))
        if_error = True
    else:
        print (f"{blank}vendorid Test Pass ({str(data['xontinfo'][0]['vendorid'])})",file = open(t_file, '+a'))
    # Check vendorversion
    temp = Find_title(out,'Vendor version',1)
    if(temp == None):
        print (f'{blank}"Vendor version" Not Found in return msg Test Fail ',file = open(t_file, '+a'))
        if_error = True
    elif(str(data['xontinfo'][0]['vendorversion']) != str(temp)):
        print (f"{blank}vendorversion Test Fail ({data['xontinfo'][0]['vendorversion']}!={temp})",file = open(t_file, '+a'))
        if_error = True
    else:
        print (f"{blank}vendorversion Test Pass ({str(data['xontinfo'][0]['vendorversion'])})",file = open(t_file, '+a'))
    # Check bootversion
    temp = Find_title(out,'V V',3)
    if(temp == None):
        print (f'{blank}"V V" Not Found in return msg, bootversion Test Fail ',file = open(t_file, '+a'))
        if_error = True
    elif(str(data['xontinfo'][0]['bootversion']) != str(temp)):
        print (f"{blank}bootversion Test Fail ({data['xontinfo'][0]['bootversion']}!={temp})",file = open(t_file, '+a'))
        if_error = True
    else:
        print (f"{blank}bootversion Test Pass ({str(data['xontinfo'][0]['bootversion'])})",file = open(t_file, '+a'))
    # Check fwversiona
    temp = Find_title(out,'FW version 1',1)
    if(temp == None):
        print (f'{blank}"FW version 1" Not Found in return msg Test Fail ',file = open(t_file, '+a'))
        if_error = True
    elif(str(data['xontinfo'][0]['fwversiona']) != str(temp)):
        print (f"{blank}fwversiona Test Fail ({data['xontinfo'][0]['fwversiona']}!={temp})",file = open(t_file, '+a'))
        if_error = True
    else:
        print (f"{blank}fwversiona Test Pass ({str(data['xontinfo'][0]['fwversiona'])})",file = open(t_file, '+a'))
    # Check fwversionb
    temp = Find_title(out,'FW version 2',1)
    if(temp == None):
        print (f'{blank}"FW version 2" Not Found in return msg Test Fail ',file = open(t_file, '+a'))
        if_error = True
    elif(str(data['xontinfo'][0]['fwversionb']) != str(temp)):
        print (f"{blank}fwversionb Test Fail ({data['xontinfo'][0]['fwversionb']}!={temp})",file = open(t_file, '+a'))
        if_error = True
    else:
        print (f"fwversionb Test Pass ({str(data['xontinfo'][0]['fwversionb'])})",file = open(t_file, '+a'))
    # Check distance
    temp = Find_title(out,'Estimated distance',1)
    if(temp == None):
        print (f'{blank}"Estimated distance" Not Found in return msg Test Fail ',file = open(t_file, '+a'))
        if_error = True
    elif(str(data['xontinfo'][0]['distance']) != str(temp)):
        print (f"{blank}Estimated distance Test Fail ({data['xontinfo'][0]['distance']}!={temp})",file = open(t_file, '+a'))
        if_error = True
    else:
        print (f"{blank}Estimated distance Test Pass ({str(data['xontinfo'][0]['distance'])})",file = open(t_file, '+a'))
    # Check ontrowstatus
    print (f"{blank}...ontrowstatus...",file = open(t_file, '+a'))

    # Check templatename
    out = f('sh in remote x 17-1-128 config',tnet_hndl,output_file)              ###
    temp = Find_title(out,'template',2)
    if(temp == None):
        print (f'{blank}"template" Not Found in return msg Test Fail ',file = open(t_file, '+a'))
        if_error = True
    elif(str(data['xontinfo'][0]['templatename']) != str(temp)):
        print (f"{blank}templatename Test Fail ({data['xontinfo'][0]['templatename']}!={temp})",file = open(t_file, '+a'))
        if_error = True
    else:
        print (f"{blank}templatename Test Pass ({str(data['xontinfo'][0]['templatename'])})",file = open(t_file, '+a'))
    # Check serialNumber
    temp = Find_title(out,'serialNumber',2)
    if(temp == None):
        print (f'{blank}"serialNumber" Not Found in return msg Test Fail ',file = open(t_file, '+a'))
        if_error = True
    elif(str(data['xontinfo'][0]['acturalsn']) != str(temp)):
        print (f"{blank}serialNumber Test Fail ({data['xontinfo'][0]['acturalsn']}!={temp})",file = open(t_file, '+a'))
        if_error = True
    else:
        print (f"{blank}serialNumber Test Pass ({str(data['xontinfo'][0]['acturalsn'])})",file = open(t_file, '+a'))
    # Check registration-id
    temp = Find_title(out,'registration-id',2)
    if(temp == None):
        print (f'{blank}"registration-id" Not Found in return msg Test Fail ',file = open(t_file, '+a'))
        if_error = True
    elif(str(data['xontinfo'][0]['acturalpassword']) != str(temp)):
        print (f"{blank}registration-id Test Fail ({data['xontinfo'][0]['acturalpassword']}!={temp})",file = open(t_file, '+a'))
        if_error = True
    else:
        print (f"{blank}registration-id Test Pass ({str(data['xontinfo'][0]['acturalpassword'])})",file = open(t_file, '+a'))
    # Check description
    temp = Find_title(out,'description',2)
    if(temp == None):
        print (f'{blank}"description" Not Found in return msg Test Fail ',file = open(t_file, '+a'))
        if_error = True
    elif(str(data['xontinfo'][0]['description']) != str(temp)):
        print (f"{blank}description Test Fail ({data['xontinfo'][0]['description']}!={temp})",file = open(t_file, '+a'))
        if_error = True
    else:
        print (f"{blank}description Test Pass ({str(data['xontinfo'][0]['description'])})",file = open(t_file, '+a'))
    
    # (xontinfo - waninfo)    
    print (f'\n{blank}waninfo ',file = open(t_file, '+a'))
    out = f('sh in remote x 17-1-128 wan-status',tnet_hndl,output_file)              ###    
    # i = 0
    i = 1 # Wan 2
    # Check Enable
    dict_w = {"Enable":"1","Disable":"0"}
    temp = Find_title(out,f'Wan {i+1}',2).split(' ')[1]
    if(temp == None):
        print (f'{blank}"Wan {i+1}" Not Found in return msg, Enable Test Fail ',file = open(t_file, '+a'))
        if_error = True
    elif(str(data['xontinfo'][0]['waninfo'][i]['enable']) != dict_w[temp]):
        print (f"{blank}Wan {i+1} Enable Test Fail ({data['xontinfo'][0]['waninfo'][i]['enable']}!={dict_w[temp]} {temp})",file = open(t_file, '+a'))
        if_error = True
    else:
        print (f"{blank}Wan {i+1} Enable Test Pass ({str(data['xontinfo'][0]['waninfo'][i]['enable'])} {temp})",file = open(t_file, '+a'))
    # Check type
    temp = Find_title(out,'Connection Type',2).split(' ')[1]
    if(temp == None):
        print (f'{blank}"Connection Type" Not Found in return msg Test Fail ',file = open(t_file, '+a'))
        if_error = True
    elif(str(data['xontinfo'][0]['waninfo'][i]['type']) != temp):
        print (f"{blank}Wan {i+1} Connection Type Test Fail ({data['xontinfo'][0]['waninfo'][i]['type']}!={temp})",file = open(t_file, '+a'))
        if_error = True
    else:
        print (f"{blank}Wan {i+1} Connection Type Test Pass ({str(data['xontinfo'][0]['waninfo'][i]['type'])})",file = open(t_file, '+a'))
    # Check linkstate
    temp = Find_title(out,'Status',2).split(' ')[1]
    if(temp == None):
        print (f'{blank}"Status" Not Found in return msg, linkstate Test Fail ',file = open(t_file, '+a'))
        if_error = True
    elif(str(data['xontinfo'][0]['waninfo'][i]['linkstate']) != temp):
        print (f"{blank}Wan {i+1} linkstate Test Fail ({data['xontinfo'][0]['waninfo'][i]['linkstate']}!={temp})",file = open(t_file, '+a'))
        if_error = True
    else:
        print (f"{blank}Wan {i+1} linkstate Test Pass ({str(data['xontinfo'][0]['waninfo'][i]['linkstate'])})",file = open(t_file, '+a'))
    # Check servicetype
    temp = Find_title(out,'Service',2).split(' ')[1]
    if(temp == None):
        print (f'{blank}"Service" Not Found in return msg, servicetype Test Fail ',file = open(t_file, '+a'))
        if_error = True
    elif(str(data['xontinfo'][0]['waninfo'][i]['servicetype']) != temp):
        print (f"{blank}Wan {i+1} servicetype Test Fail ({data['xontinfo'][0]['waninfo'][i]['servicetype']}!={temp})",file = open(t_file, '+a'))
        if_error = True
    else:
        print (f"{blank}Wan {i+1} servicetype Test Pass ({str(data['xontinfo'][0]['waninfo'][i]['servicetype'])})",file = open(t_file, '+a'))
    # Check vlan
    temp = Find_title(out,'Vlan',2).split(' ')[1]
    if(temp == None):
        print (f'{blank}"Vlan" Not Found in return msg Test Fail ',file = open(t_file, '+a'))
        if_error = True
    elif(str(data['xontinfo'][0]['waninfo'][i]['vlan']) != temp):
        print (f"{blank}Wan {i+1} Vlan Test Fail ({data['xontinfo'][0]['waninfo'][i]['vlan']}!={temp})",file = open(t_file, '+a'))
        if_error = True
    else:
        print (f"{blank}Wan {i+1} Vlan Test Pass ({str(data['xontinfo'][0]['waninfo'][i]['vlan'])})",file = open(t_file, '+a'))
    # Check priority
    temp = Find_title(out,'Priority',2).split(' ')[1]
    if(temp == None):
        print (f'{blank}"Priority" Not Found in return msg Test Fail ',file = open(t_file, '+a'))
        if_error = True
    elif(str(data['xontinfo'][0]['waninfo'][i]['priority']) != temp):
        print (f"{blank}Wan {i+1} Priority Test Fail ({data['xontinfo'][0]['waninfo'][i]['priority']}!={temp})",file = open(t_file, '+a'))
        if_error = True
    else:
        print (f"{blank}Wan {i+1} Priority Test Pass ({str(data['xontinfo'][0]['waninfo'][i]['priority'])})",file = open(t_file, '+a'))
    # Check ipmode ...
    print (f'{blank}...ipmode... ',file = open(t_file, '+a'))
    # Check ip
    temp = Find_title(out,'IP address',2).split(' ')[1]
    if(temp == None):
        print (f'{blank}"IP address" Not Found in return msg Test Fail ',file = open(t_file, '+a'))
        if_error = True
    elif(str(data['xontinfo'][0]['waninfo'][i]['ip']) != temp):
        print (f"{blank}Wan {i+1} IP address Test Fail ({data['xontinfo'][0]['waninfo'][i]['ip']}!={temp})",file = open(t_file, '+a'))
        if_error = True
    else:
        print (f"{blank}Wan {i+1} IP address Test Pass ({str(data['xontinfo'][0]['waninfo'][i]['ip'])})",file = open(t_file, '+a'))
    # Check mask
    temp = Find_title(out,'IP Mask',2).split(' ')[1]
    if(temp == None):
        print (f'{blank}"IP Mask" Not Found in return msg Test Fail ',file = open(t_file, '+a'))
        if_error = True
    elif(str(data['xontinfo'][0]['waninfo'][i]['mask']) != temp):
        print (f"{blank}Wan {i+1} IP Mask Test Fail ({data['xontinfo'][0]['waninfo'][i]['mask']}!={temp})",file = open(t_file, '+a'))
        if_error = True
    else:
        print (f"{blank}Wan {i+1} IP Mask Test Pass ({str(data['xontinfo'][0]['waninfo'][i]['mask'])})",file = open(t_file, '+a'))
    # Check gateway
    temp = Find_title(out,'Gateway',2).split(' ')[1]
    if(temp == None):
        print (f'{blank}"Gateway" Not Found in return msg Test Fail ',file = open(t_file, '+a'))
        if_error = True
    elif(str(data['xontinfo'][0]['waninfo'][i]['gateway']) != temp):
        print (f"{blank}Wan {i+1} Gateway Test Fail ({data['xontinfo'][0]['waninfo'][i]['gateway']}!={temp})",file = open(t_file, '+a'))
        if_error = True
    else:
        print (f"{blank}Wan {i+1} Gateway Test Pass ({str(data['xontinfo'][0]['waninfo'][i]['gateway'])})",file = open(t_file, '+a'))
    # Check primarydns
    temp = Find_title(out,'Primary DNS',2).split(' ')[1]
    if(temp == None):
        print (f'{blank}"Primary DNS" Not Found in return msg Test Fail ',file = open(t_file, '+a'))
        if_error = True
    elif(str(data['xontinfo'][0]['waninfo'][i]['primarydns']) != temp):
        print (f"{blank}Wan {i+1} Primary DNS Test Fail ({data['xontinfo'][0]['waninfo'][i]['primarydns']}!={temp})",file = open(t_file, '+a'))
        if_error = True
    else:
        print (f"{blank}Wan {i+1} Primary DNS Test Pass ({str(data['xontinfo'][0]['waninfo'][i]['primarydns'])})",file = open(t_file, '+a'))
    # Check secondarydns
    temp = Find_title(out,'Secondary DNS',2).split(' ')[1]
    if(temp == None):
        print (f'{blank}"Secondary DNS" Not Found in return msg Test Fail ',file = open(t_file, '+a'))
        if_error = True
    elif(str(data['xontinfo'][0]['waninfo'][i]['secondarydns']) != temp):
        print (f"{blank}Wan {i+1} Secondary DNS Test Fail ({data['xontinfo'][0]['waninfo'][i]['secondarydns']}!={temp})",file = open(t_file, '+a'))
        if_error = True
    else:
        print (f"{blank}Wan {i+1} Secondary DNS Test Pass ({str(data['xontinfo'][0]['waninfo'][i]['secondarydns'])})",file = open(t_file, '+a'))
    # Check nat ...
    print (f'{blank}...nat... ',file = open(t_file, '+a'))

    for i in range(0,8):
        if(i==1):
            continue
        # Check Disable
        temp = Find_title(out,f'Wan {i+1}',2).split(' ')[1]
        if(temp == None):
            print (f'{blank}"Wan {i+1}" Not Found in return msg, Disable Test Fail ',file = open(t_file, '+a'))
            if_error = True
        elif(str(data['xontinfo'][0]['waninfo'][i]['enable']) != dict_w[temp]):
            print (f"{blank}Wan {i+1} Enable Test Fail ({data['xontinfo'][0]['waninfo'][i]['enable']}!={dict_w[temp]} {temp})",file = open(t_file, '+a'))
            if_error = True
        else:
            print (f"{blank}Wan {i+1} Enable Test Pass ({str(data['xontinfo'][0]['waninfo'][i]['enable'])} {temp})",file = open(t_file, '+a'))
    if(if_error == False):
        print (f"[MSAN1-16430][Device_info - Interface - xontinfo] Result Pass",file = open(t_file, '+a'))
    else:
        print (f"[MSAN1-16430][Device_info - Interface - xontinfo] Result Fail",file = open(t_file, '+a'))

    if_error = False
    if(if_error == False):
        print (f"[MSAN1-16431][Device_info - Interface – hwinfo] Result Pass",file = open(t_file, '+a'))
    else:
        print (f"[MSAN1-16431][Device_info - Interface – hwinfo] Result Fail",file = open(t_file, '+a'))



except Exception as e:
    tnet_hndl.close() 
    print(e)
    raise Exception
# out = f('show lc st 17',tnet_hndl,output_file)              ###
# data['slotinfo'][16]['slot']
# data['slotinfo'][16]['type']
# data['slotinfo'][16]['operstatus']
# data['slotinfo'][16]['adminstate']
# data['slotinfo'][16]['alarmstate']
# data['slotinfo'][16]['hwversion']
# data['slotinfo'][16]['fwversion']
# data['slotinfo'][16]['serialnumber']


try:
    tnet_hndl.close()
except Exception as e:
    print(e)
    raise Exception

00
[{'interface': 'inband', 'ip': '192.168.169.86', 'mask': '255.255.255.0', 'gateway': '192.168.169.254', 'vid': 7}, {'interface': 'outband', 'ip': '192.168.0.1', 'mask': '255.255.255.0', 'gateway': '0.0.0.0', 'vid': 0}]
inband
'NoneType' object has no attribute 'split'


Exception: 

In [2]:
print(temp)

None
